# Búsqueda A* — 8-Puzzle

Lee `estadoinicial.txt` y `estadofinal.txt`. El vacío se representa con **0**.

## Heurística inventada: *Eco decimal de columnas*

Cada columna del tablero se lee de arriba hacia abajo como un número de tres dígitos.  
Por ejemplo, la columna izquierda de:

```
3 2 6
5 0 4
1 8 7
```

vale **351**.

La heurística compara esas tres cifras con las del estado meta:

$$
h = \frac{|C_0 - C_0^*| + |C_1 - C_1^*| + |C_2 - C_2^*|}{100}
$$

Si las columnas se parecen a las del objetivo, $h$ baja.  
A* usa $f = g + h$, donde $g$ es la cantidad de movimientos hechos.

In [ ]:
import heapq

## Lectura y utilidades

In [ ]:
def leer_estado(ruta):
    """Lee un tablero 3x3. Cada línea es como 326."""
    tablero = []
    with open(ruta) as f:
        for linea in f:
            linea = linea.strip()
            if linea:
                tablero.append(tuple(int(c) for c in linea))
    return tuple(tablero)


def mostrar(estado):
    for fila in estado:
        print(*fila)
    print()


def buscar_cero(estado):
    for i in range(3):
        for j in range(3):
            if estado[i][j] == 0:
                return i, j

## Heurística

In [ ]:
def heuristica(estado, meta):
    """Eco decimal de columnas."""
    h = 0
    for j in range(3):
        col_act = estado[0][j] * 100 + estado[1][j] * 10 + estado[2][j]
        col_meta = meta[0][j] * 100 + meta[1][j] * 10 + meta[2][j]
        h += abs(col_act - col_meta)
    return h / 100

## Vecinos y A*

In [ ]:
# (di, dj, nombre del movimiento del hueco)
DIRS = [(-1, 0, "Arriba"), (1, 0, "Abajo"), (0, -1, "Izquierda"), (0, 1, "Derecha")]


def vecinos(estado):
    """Devuelve pares (nuevo_estado, nombre_movimiento)."""
    i, j = buscar_cero(estado)
    resultado = []
    for di, dj, nombre in DIRS:
        ni, nj = i + di, j + dj
        if 0 <= ni < 3 and 0 <= nj < 3:
            nuevo = [list(fila) for fila in estado]
            nuevo[i][j], nuevo[ni][nj] = nuevo[ni][nj], nuevo[i][j]
            resultado.append((tuple(tuple(f) for f in nuevo), nombre))
    return resultado


def a_estrella(inicio, meta):
    """Retorna (movimientos, iteraciones) o (None, iteraciones)."""
    # cola: (f, g, estado)
    cola = [(heuristica(inicio, meta), 0, inicio)]
    vino_de = {inicio: None}   # estado -> (padre, movimiento)
    costo_g = {inicio: 0}
    iteraciones = 0

    while cola:
        f, g, actual = heapq.heappop(cola)
        iteraciones += 1

        if actual == meta:
            # reconstruir camino
            movs = []
            e = actual
            while vino_de[e] is not None:
                padre, mov = vino_de[e]
                movs.append(mov)
                e = padre
            movs.reverse()
            return movs, iteraciones

        if g > costo_g[actual]:
            continue

        for nxt, mov in vecinos(actual):
            nuevo_g = g + 1
            if nxt not in costo_g or nuevo_g < costo_g[nxt]:
                costo_g[nxt] = nuevo_g
                vino_de[nxt] = (actual, mov)
                heapq.heappush(cola, (nuevo_g + heuristica(nxt, meta), nuevo_g, nxt))

    return None, iteraciones

## Ejecución

In [ ]:
inicio = leer_estado("estadoinicial.txt")
meta = leer_estado("estadofinal.txt")

print("Estado inicial:")
mostrar(inicio)
print("Estado final:")
mostrar(meta)
print("h(inicio) =", heuristica(inicio, meta))

In [ ]:
movimientos, iteraciones = a_estrella(inicio, meta)

if movimientos is None:
    print("No se encontró solución.")
    print("Iteraciones:", iteraciones)
else:
    print("Iteraciones:", iteraciones)
    print("Cantidad de movimientos:", len(movimientos))
    print()
    print("Movimientos del hueco:")
    for i, m in enumerate(movimientos, 1):
        print(f"  {i}. {m}")

    # mostrar tableros paso a paso
    print()
    print("Trayectoria:")
    estado = inicio
    print("Paso 0")
    mostrar(estado)
    for i, m in enumerate(movimientos, 1):
        for nxt, nombre in vecinos(estado):
            if nombre == m:
                estado = nxt
                break
        print(f"Paso {i}: {m}")
        mostrar(estado)